# No.4 離散フーリエ変換 — 全課題パイプライン

1. 1Dデータモデル g(x) を生成
2. DFT（数値積分）を実装・実行
3. DFT結果をIFFTで再構成
4. パラメータ (N, Dx) を変えて9パターン実行し比較

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import japanize_matplotlib
import scipy.io
import os

os.makedirs("output", exist_ok=True)

m = 1024
dx = 0.025

## 課題1: 1Dデータモデル g(x) 生成

In [ ]:
x = (np.arange(m) - m / 2) * dx
g = np.zeros(m)
for i in range(m):
    xi = x[i]
    if ((-6.0 <= xi < -3.5) or (0.2 <= xi < 2.0) or (2.5 <= xi <= 6.0)):
        g[i] = 0.1
    elif -3.5 <= xi < -2.0:
        g[i] = 0.7
    elif -2.0 <= xi < 0:
        g[i] = 0.8
    elif 0 <= xi < 0.2:
        g[i] = 0.5
    elif 2.0 <= xi < 2.5:
        g[i] = 0.2

scipy.io.savemat(f"m{m}dx{dx*1000:.0f}.mat", {"Signal": g})

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(x, g, linewidth=1.5)
ax.set_xlabel("x [cm]")
ax.set_ylabel("g(x)")
ax.set_title(f"1D Data Model (m={m}, dx={dx})")
ax.grid(True)
fig.tight_layout()
fig.savefig("output/kadai1_1Ddata.png", dpi=150)
plt.show()

## 課題2-4: DFT → IFFT再構成（N × Dx の9パターン）

DFT式（数値積分版）:

$$G(n\delta f) = \delta x \sum_{i=0}^{m-1} g(i\delta x)\cos\theta - j\delta x \sum_{i=0}^{m-1} g(i\delta x)\sin\theta$$

$$\theta = 2\pi(n-N/2)\Delta f(i-m/2)\delta x, \quad \Delta f = \frac{1}{N\Delta x}$$

In [ ]:
def dft_numerical(g, m, dx, N, Dx):
    """数値積分による離散フーリエ変換（課題の式に忠実）"""
    Df = 1.0 / (N * Dx)
    re = np.zeros(N)
    im = np.zeros(N)
    for n in range(N):
        n_shift = n - N / 2
        for i in range(m):
            i_shift = i - m / 2
            theta = 2 * np.pi * (n_shift * Df) * (i_shift * dx)
            re[n] += g[i] * np.cos(theta)
            im[n] -= g[i] * np.sin(theta)
        re[n] *= dx
        im[n] *= dx
    return re + 1j * im

# パラメータ組み合わせ
N_list = [128, 256, 512]
Dx_list = [0.05, 0.1, 0.5]

results = {}
for N in N_list:
    for Dx in Dx_list:
        print(f"DFT: N={N}, Dx={Dx} ... ", end="", flush=True)
        G = dft_numerical(g, m, dx, N, Dx)
        
        # IFFT再構成
        recon = np.fft.fftshift(np.fft.ifft(np.fft.fftshift(G)))
        
        results[(N, Dx)] = (G, recon)
        
        param_name = f"m{m}dx{dx*1000:.0f}N{N}Dx{Dx*1000:.0f}"
        scipy.io.savemat(f"output/{param_name}_DFT.mat", {"Signal": G})
        print("done")

print("全9パターンのDFT完了")

### DFT結果とIFFT再構成の比較プロット（9パターン）

In [ ]:
for N in N_list:
    for Dx in Dx_list:
        G, recon = results[(N, Dx)]
        param_name = f"m{m}dx{dx*1000:.0f}N{N}Dx{Dx*1000:.0f}"

        fig, axes = plt.subplots(1, 2, figsize=(14, 4))

        # DFT結果
        axes[0].plot(G.real, linewidth=1, label="real part")
        axes[0].plot(G.imag, linewidth=1, label="imaginary part")
        axes[0].set_title(f"{param_name} DFT")
        axes[0].set_xlabel("Data")
        axes[0].set_ylabel("Amplitude")
        axes[0].legend()
        axes[0].grid(True)

        # IFFT再構成
        axes[1].plot(recon.real, linewidth=1, label="real part")
        axes[1].plot(recon.imag, linewidth=1, label="imaginary part")
        axes[1].set_title(f"{param_name} DFT→IFFT")
        axes[1].set_xlabel("Data")
        axes[1].set_ylabel("Amp.")
        axes[1].legend()
        axes[1].grid(True)

        fig.tight_layout()
        fig.savefig(f"output/{param_name}.png", dpi=150)
        plt.show()